# Wi-Fi basics and wireless security protocols (WPA2/WPA3)  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 3.2.a: Wi-Fi configuration security evaluator

**Skill:** Mitigate Risk · **Maps to:** 3.2.A · **Language:** Python 3 (pytest)

## Problem
You are building an automated Wi-Fi configuration checker for Crestfield Analytics, mirroring what
enterprise wireless assessment tools do. Given a dictionary describing one access point's
configuration, return the sorted list of security gap codes that configuration has, based on the
criteria from the lesson (protocol strength, PSK password length, guest network isolation, admin
interface password strength, and firmware currency).

## Function contract
Implement `evaluate_wifi_config(cfg)` in the code editor (see the Reference section below).

## Input
- `cfg`: `dict` with:
  - `"protocol"`: `str`, one of `"WPA3"`, `"WPA2-AES"`, `"WPA2-TKIP"`, `"WEP"`, `"OPEN"`
  - `"mode"`: `str`, `"personal"` or `"enterprise"`
  - `"psk_length"`: `int`, length of the pre-shared key/password (only meaningful when
    `mode == "personal"`)
  - `"guest_network_isolated"`: `bool`, whether guest devices are on a separate, isolated SSID/VLAN
  - `"admin_password_strong"`: `bool`, whether the AP's management interface has a strong unique
    password
  - `"firmware_current"`: `bool`, whether the AP is running current (patched) firmware

## Output
A sorted `list[str]` of gap codes present in the configuration (empty list if there are no gaps).
Possible gap codes:
- `"insecure_protocol"`, protocol is `"WEP"`, `"WPA2-TKIP"`, or `"OPEN"`
- `"weak_psk"`, `mode == "personal"` and `psk_length < 20`
- `"no_guest_isolation"`, `guest_network_isolated` is `False`
- `"weak_admin_password"`, `admin_password_strong` is `False`
- `"outdated_firmware"`, `firmware_current` is `False`

## Examples
```python
cfg = {
    "protocol": "WPA2-TKIP", "mode": "personal", "psk_length": 12,
    "guest_network_isolated": True, "admin_password_strong": True, "firmware_current": True,
}
evaluate_wifi_config(cfg)
# -> ["insecure_protocol", "weak_psk"]

cfg2 = {
    "protocol": "WPA3", "mode": "enterprise", "psk_length": 0,
    "guest_network_isolated": True, "admin_password_strong": True, "firmware_current": True,
}
evaluate_wifi_config(cfg2)
# -> []
```

## Constraints
- Deterministic, offline, standard library only.
- `psk_length` is only checked when `mode == "personal"`; ignore it entirely for `"enterprise"`
  mode (enterprise mode uses per-user 802.1X credentials, not a shared PSK).
- Gap codes in the output list must be sorted alphabetically and contain no duplicates.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def evaluate_wifi_config(cfg: dict) -> list[str]
```

### Input (`cfg`)
`dict` with:
- `"protocol"`: `str`, one of `"WPA3"`, `"WPA2-AES"`, `"WPA2-TKIP"`, `"WEP"`, `"OPEN"`
- `"mode"`: `str`, `"personal"` or `"enterprise"`
- `"psk_length"`: `int`, PSK/password length (meaningful only when `mode == "personal"`)
- `"guest_network_isolated"`: `bool`
- `"admin_password_strong"`: `bool`
- `"firmware_current"`: `bool`

### Output (`result`)
A sorted `list[str]` (no duplicates) of gap codes:
- `"insecure_protocol"`, `protocol` in `{"WEP", "WPA2-TKIP", "OPEN"}`
- `"weak_psk"`, `mode == "personal"` and `psk_length < 20`
- `"no_guest_isolation"`, `guest_network_isolated is False`
- `"weak_admin_password"`, `admin_password_strong is False`
- `"outdated_firmware"`, `firmware_current is False`

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- `psk_length` is ignored (never triggers `"weak_psk"`) when `mode == "enterprise"`.
- Output is sorted alphabetically with no duplicates; empty list if no gaps apply.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 3.2.a. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from typing import Dict, List

_INSECURE_PROTOCOLS = {"WEP", "WPA2-TKIP", "OPEN"}


def evaluate_wifi_config(cfg: Dict) -> List[str]:
    """Return the sorted list of security gap codes in a Wi-Fi AP configuration.

    Rules (see the Reference section below):
      - insecure_protocol: protocol in {WEP, WPA2-TKIP, OPEN}
      - weak_psk: mode == "personal" and psk_length < 20 (ignored for enterprise mode)
      - no_guest_isolation: guest_network_isolated is False
      - weak_admin_password: admin_password_strong is False
      - outdated_firmware: firmware_current is False
    """
    gaps = []

    if cfg["protocol"] in _INSECURE_PROTOCOLS:
        gaps.append("insecure_protocol")

    if cfg["mode"] == "personal" and cfg["psk_length"] < 20:
        gaps.append("weak_psk")

    if not cfg["guest_network_isolated"]:
        gaps.append("no_guest_isolation")

    if not cfg["admin_password_strong"]:
        gaps.append("weak_admin_password")

    if not cfg["firmware_current"]:
        gaps.append("outdated_firmware")

    return sorted(gaps)


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def _cfg(protocol="WPA3", mode="personal", psk_length=24, guest=True, admin=True, fw=True):
    return {
        "protocol": protocol,
        "mode": mode,
        "psk_length": psk_length,
        "guest_network_isolated": guest,
        "admin_password_strong": admin,
        "firmware_current": fw,
    }


# ---- visible cases -------------------------------------------------------

def test_perfect_config_no_gaps():
    assert impl.evaluate_wifi_config(_cfg()) == []


def test_tkip_and_weak_psk():
    cfg = _cfg(protocol="WPA2-TKIP", psk_length=12)
    assert impl.evaluate_wifi_config(cfg) == ["insecure_protocol", "weak_psk"]


def test_wep_flagged():
    cfg = _cfg(protocol="WEP")
    assert impl.evaluate_wifi_config(cfg) == ["insecure_protocol"]


def test_open_flagged():
    cfg = _cfg(protocol="OPEN")
    assert impl.evaluate_wifi_config(cfg) == ["insecure_protocol"]


def test_wpa2_aes_acceptable():
    cfg = _cfg(protocol="WPA2-AES", psk_length=20)
    assert impl.evaluate_wifi_config(cfg) == []


def test_no_guest_isolation_flagged():
    cfg = _cfg(guest=False)
    assert impl.evaluate_wifi_config(cfg) == ["no_guest_isolation"]


def test_weak_admin_password_flagged():
    cfg = _cfg(admin=False)
    assert impl.evaluate_wifi_config(cfg) == ["weak_admin_password"]


def test_outdated_firmware_flagged():
    cfg = _cfg(fw=False)
    assert impl.evaluate_wifi_config(cfg) == ["outdated_firmware"]


def test_all_gaps_sorted():
    cfg = _cfg(protocol="WEP", psk_length=5, guest=False, admin=False, fw=False)
    assert impl.evaluate_wifi_config(cfg) == [
        "insecure_protocol",
        "no_guest_isolation",
        "outdated_firmware",
        "weak_admin_password",
        "weak_psk",
    ]


# ---- hidden / edge cases -------------------------------------------------

def test_enterprise_mode_ignores_psk_length():
    """Enterprise mode uses per-user 802.1X creds; psk_length must never trigger weak_psk."""
    cfg = _cfg(mode="enterprise", psk_length=1)
    assert impl.evaluate_wifi_config(cfg) == []


def test_psk_length_boundary_19_is_weak():
    cfg = _cfg(psk_length=19)
    assert impl.evaluate_wifi_config(cfg) == ["weak_psk"]


def test_psk_length_boundary_20_is_ok():
    cfg = _cfg(psk_length=20)
    assert impl.evaluate_wifi_config(cfg) == []


def test_wpa3_enterprise_perfect():
    cfg = _cfg(protocol="WPA3", mode="enterprise", psk_length=0)
    assert impl.evaluate_wifi_config(cfg) == []


def test_wpa2_tkip_enterprise_still_flags_protocol_only():
    cfg = _cfg(protocol="WPA2-TKIP", mode="enterprise", psk_length=0)
    assert impl.evaluate_wifi_config(cfg) == ["insecure_protocol"]


def test_no_duplicate_gap_codes():
    cfg = _cfg(protocol="OPEN", psk_length=0, guest=False, admin=False, fw=False)
    result = impl.evaluate_wifi_config(cfg)
    assert len(result) == len(set(result))


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
